# COMP9085 Week 5 Laboratory
## Naive Bayes on real food-recall text: a complete pipeline

**Two hours.** Naive Bayes is a simple model, but it is a complete
machine-learning pipeline. In this lab you will build one from raw text to an
evaluated, documented result, on real data, and then question what it learned.

**Data.** A frozen snapshot of the U.S. FDA's openFDA food enforcement reports
(export dated 2026-09-28, public domain), in `data/openfda_food_recall_events.csv`
next to this notebook. One row represents an **event–reason–classification
record**: one recall event, one reason text, one FDA classification. Most
event–reason pairs have one classification, but a few have more than one (Task 1).
The target is `is_class_I`: 1 if FDA classified the recall as Class I
(a reasonable probability of serious adverse health consequences or death),
0 for Class II or III. This is real data used for classroom evaluation. The
label is an administrative decision, not a measured health outcome, and nothing
in this lab is a food-safety tool. `data/README.md` documents the source and
how the snapshot was made.

**No network is needed.** Everything runs from the CSV.

**Structure.** **Tasks 1–8 are the core lab.** A checkpoint after Task 8 marks
the end of the core. Two extensions follow; both are optional.

**The Week 5 page is your interactive companion.**

| Task | Interactive on the Week 5 page |
|---|---|
| 1–2 | Section 4 (what is one row?) and explorer D |
| 3 | Explorer C: text representation |
| 4 | The classifier explorer (it reproduces your Task 4 numbers) |
| 5 | Explorer B and the probe in the classifier explorer |
| 6 | The classifier explorer, Errors view |
| 7 | Explorer D: leakage |
| 8 | The classifier explorer, Probabilities view |
| Extensions | Explorer E: model comparison |

**Deliverable.** This notebook, run top to bottom, with every question answered
in the markdown cells.

### Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, GridSearchCV, train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve

SEED = 42
df = pd.read_csv("data/openfda_food_recall_events.csv", keep_default_na=False)
print(df.shape)
df.head()

A helper used throughout: every model in this lab is evaluated with the
same function, at the same threshold (positive if p ≥ 0.5).

In [ ]:
def evaluate(y_true, p, name):
    y_pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"model": name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, p) if len(set(p)) > 1 else 0.5,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}

## Task 1 — What is one row?

The FDA export has one record per recalled **product**. A single recall
**event** can cover many products with the same reason text; the column
`n_product_records` says how many product records shared each row's text. Rows
are distinct (`event_id`, `reason_for_recall`, `classification`) combinations.

Compute: the number of rows, the number of distinct recall events, the total
number of product records, the largest number of product records in a single
event, the counts of each FDA class, and the fraction of rows that are Class I.
Then count the distinct event–reason pairs, how many of them carry more than one
FDA classification, and how many disagree on the target `is_class_I`.
Then print five reason texts with their class (use `random_state=SEED`).

In [ ]:
# your code here


**Question 1.1.** Why is one row an event–reason–classification record, and
not a product record? What would go wrong if you trained and tested on product
records?

**Question 1.2.** The target is FDA's hazard classification. Name one thing it
measures and one thing it does not.

**Question 1.3.** Some event–reason pairs appear with more than one
classification. Should they be removed? What do they tell you about the task?

*Your answer here.*

## Task 2 — Split by recall event, and set a baseline

The split below keeps every recall event entirely on one side
(`StratifiedGroupKFold`, 4 folds, first fold used as the test set, groups =
`event_id`). It is the split used on the website and in the lecture, so your
numbers can be compared with theirs. Run it, then:

1. check that no `event_id` appears in both `train` and `test`;
2. report the sizes and the Class I fraction of each side;
3. fit a `DummyClassifier(strategy="most_frequent")` and evaluate it on the
   test set with `evaluate`.

In [ ]:
sgk = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgk.split(df.reason_for_recall, df.is_class_I, groups=df.event_id))
train, test = df.iloc[train_idx], df.iloc[test_idx]
X_train, y_train = train.reason_for_recall, train.is_class_I.values
X_test, y_test = test.reason_for_recall, test.is_class_I.values

In [ ]:
# your code here


**Question 2.1.** What does the baseline's accuracy mean, and why is its F1
for Class I zero? Any model you build must be compared with this number.

*Your answer here.*

## Task 3 — Turn text into numbers

Fit a `CountVectorizer()` with its default settings **on the training text
only**. Then:

1. report the size of the vocabulary;
2. transform the first training text and print the words it contains with their
   counts (`vec.inverse_transform` or the non-zero columns of the row);
3. transform the new text `"Product contains undeclared lupin flour."` and say
   which of its words survive.

Explorer C on the Week 5 page does the same thing for three texts at a time.

In [ ]:
# your code here


**Question 3.1.** What happened to "lupin", and why? What does this mean
for a recall that describes a hazard in words the training data never used?

**Question 3.2.** Why must the vectorizer be fitted on the training text only?

*Your answer here.*

## Task 4 — Build the pipeline and evaluate it

Build a `Pipeline` of `CountVectorizer()` and `MultinomialNB(alpha=1.0)` named
`model`, fit it on the training rows, compute `p = model.predict_proba(X_test)[:, 1]`,
and add its evaluation to `results`. Print the results table (baseline and
naive Bayes).

In [ ]:
# your code here


**Question 4.1.** Summarise the result in two sentences, relative to the
baseline. Which error is more common, missed Class I texts or false alarms?

**Question 4.2.** Check your numbers against the classifier explorer on the
Week 5 page at alpha = 1. Why do they agree exactly?

*Your answer here.*

## Task 5 — Open the model

Naive Bayes stores everything it learned in two arrays: `class_log_prior_`
(log P(class)) and `feature_log_prob_` (log P(word | class), one row per class).

**(a)** For the test text whose `p` is closest to 0.5, compute the log-score of each class by hand,
`class_log_prior_ + counts @ feature_log_prob_.T`, turn the two scores into
P(Class I | text), and check that it equals `p[i]`.

**(b)** Compute each word's log-ratio, `feature_log_prob_[1] - feature_log_prob_[0]`.
List the 15 words that push hardest towards Class I, first among words used at
least 10 times in the training text, then among words used in at least 20
different training **events**.

In [ ]:
# your code here


**Question 5.1.** Explain in one sentence why `p[i]` can be computed with
only additions, then one subtraction and one exponential.

**Question 5.2.** Read the two word lists. What do the first describe, and what
do the second describe? What does this tell you about what a model learns?

*Your answer here.*

## Task 6 — Error analysis

Print the five most confident false positives (FDA Class II/III, highest `p`)
and the five most confident false negatives (FDA Class I, lowest `p`), with the
FDA class and the full text.

In [ ]:
# your code here


**Question 6.1.** For each group, what do the texts have in common? Could the
text alone have told a person the FDA class?

**Question 6.2.** Suggest one change to the data or the task (not the model)
that would make this classification better defined.

*Your answer here.*

## Task 7 — Leakage: the whole pipeline must not learn from the test set

**(a) Vectorizer fitted on all text.** Fit `CountVectorizer()` on **all**
`df.reason_for_recall`, then fit `MultinomialNB()` on the transformed training
rows and evaluate on the transformed test rows. How many vocabulary words occur
only in test texts?

**(b) Split by product record.** Expand the data to one row per product record,
`rows = df.loc[df.index.repeat(df.n_product_records)]`, split it at random with
`train_test_split(test_size=0.25, stratify=..., random_state=SEED)`, fit the same
pipeline, evaluate it, and report the fraction of test texts that appear word
for word in the training texts. Do the same check for the event split.

In [ ]:
# your code here


**Question 7.1.** Which mistake inflated the score, and by how much? Explain why.

**Question 7.2.** The vectorizer leak barely changed the score. Is it therefore
acceptable? Give two reasons.

**Question 7.3.** 14% of test texts appear word for word in training even with
the event split. Is that leakage?

*Your answer here.*

## Task 8 — Probabilities, and the reproducibility record

**(a)** Plot a histogram of `p` (20 bins) and report the fraction of predictions
below 0.05 or above 0.95.

**(b)** Plot a reliability diagram with `calibration_curve(y_test, p, n_bins=10)`
and the diagonal, and report the Brier score.

In [ ]:
# your code here


**Question 8.1.** A text gets `predict_proba = 0.99`. Is that a 99%
probability that the recall is Class I? Use your two plots.

**Question 8.2.** Why does naive Bayes push its scores towards 0 and 1?

*Your answer here.*

### The reproducibility record

Complete the table in the markdown cell below. Someone else should be able to
repeat your Task 4 result exactly from it.

| Item | Your entry |
|---|---|
| Data (file, source, snapshot date, rows) | |
| Unit of observation | |
| Target and class balance | |
| Split (method, seed, groups, sizes) | |
| Pipeline (every step and its settings) | |
| What was fitted on what | |
| Baseline | |
| Test-set results (accuracy, precision, recall, F1, ROC AUC, Brier) | |
| Software versions (from the cell below) | |
| One thing this result does **not** show | |

In [ ]:
import sklearn, sys
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)

*Your answer here.*

---
## ✅ CORE LAB COMPLETE

You have built a complete pipeline (data, unit of observation, split,
vectorizer, model, evaluation, error analysis, leakage checks, probabilities and
a record). The two extensions below are **optional**.

## Extension A (optional) — Other representations, and choosing alpha honestly

**(a)** Evaluate `TfidfVectorizer() + MultinomialNB()` and
`CountVectorizer(binary=True) + BernoulliNB()` on the same split and add them to
`results`.

**(b)** Choose `alpha` for the counts + MultinomialNB pipeline with
`GridSearchCV` over `[0.01, 0.03, 0.1, 0.3, 1, 3]`, using `GroupKFold(n_splits=4)`
on the **training rows only** (groups = training `event_id`, scoring = `"f1"`).
Then evaluate the chosen model once on the test set.

In [ ]:
# your code here


**Question A.1.** Which representation would you choose, and on which metric?
Why is it legitimate to choose alpha with `GridSearchCV` here, but not by
trying values and watching the test score (or the slider on the website)?

*Your answer here.*

## Extension B (optional) — A pretrained sentence encoder

A pretrained encoder maps each text to a fixed-length vector (an embedding),
which a simple classifier then uses. The module team ran this once on exactly
your split, with `sentence-transformers/all-MiniLM-L6-v2` (revision
`1110a243`, about 22.7 million parameters, a ~91 MB download) and
`LogisticRegression(C=1, max_iter=2000)` on the normalised embeddings. The stored
results are below, so **nothing needs to be downloaded**.

If you have `sentence-transformers` installed and want to reproduce them, set
`RUN_ENCODER = True`. This downloads the model once (about 91 MB) and takes
around ten seconds on a laptop CPU. It is not required.

In [ ]:
RUN_ENCODER = False   # optional: needs the sentence-transformers package and a one-time download

stored = {"accuracy": 0.7939, "precision": 0.7443, "recall": 0.7466, "f1": 0.7454, "rocAuc": 0.8636, "averagePrecision": 0.7744, "brier": 0.1474, "fractionExtreme": 0.107}
if RUN_ENCODER:
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LogisticRegression
    enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                              revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", device="cpu")
    E_train = enc.encode(list(X_train), batch_size=64, normalize_embeddings=True)
    E_test = enc.encode(list(X_test), batch_size=64, normalize_embeddings=True)
    lr = LogisticRegression(C=1.0, max_iter=2000).fit(E_train, y_train)
    results.append(evaluate(y_test, lr.predict_proba(E_test)[:, 1], "pretrained encoder + LR (run here)"))
    print(pd.DataFrame(results).round(4))
else:
    print("Stored results, same split:", stored)

**Question B.1.** Compare the encoder with naive Bayes on accuracy, F1, ROC
AUC and Brier score. Then compare them on cost, dependencies and how easily one
prediction can be explained. Which would you deploy for this task, and what
evidence would change your mind?

*Your answer here.*